# Logistic Regression

---

## Overview

**Logistic regression** is a probabilistic binary classifier. Unlike the perceptron, it outputs a *probability* in $(0, 1)$ using the sigmoid (logistic) activation function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The predicted probability is:

$$\hat{p} = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$

---

## Loss Function. Binary Cross-Entropy

$$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y^{(i)} \log \hat{p}^{(i)} + (1 - y^{(i)}) \log(1 - \hat{p}^{(i)}) \right]$$

## SGD Update Rule

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha (\hat{p} - y) \mathbf{x}$$
$$b \leftarrow b - \alpha (\hat{p} - y)$$

---

**Dataset:** Pima Indians Diabetes (`diabetes.csv`)  
**Task:** Predict diabetes diagnosis (0 = no, 1 = yes).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning import LogisticRegression
from rice_ml.preprocess import StandardScaler, train_test_split
from rice_ml.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
try:
    df = pd.read_csv('../../../data/diabetes.csv')
    X = df.drop(columns=['Outcome']).values.astype(float)
    y = df['Outcome'].values.astype(float)
    print(f'Loaded diabetes dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    X, y = data.data[:, :8], data.target.astype(float)
    print('CSV not found. using breast cancer dataset (first 8 features)')

print(f'Positive cases: {int(y.sum())} / {len(y)}')

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Train. Stochastic Gradient Descent

Each epoch processes one sample at a time and updates $\mathbf{w}$ and $b$ using the gradient of the cross-entropy loss.

In [ ]:
model = LogisticRegression(alpha=0.1, epochs=300)
model.fit(X_train, y_train)

plt.figure(figsize=(10, 6))
plt.plot(model.errors_, color='steelblue')
plt.xlabel('Epoch', fontsize=15)
plt.ylabel('Cross-Entropy Loss', fontsize=15)
plt.title('Logistic Regression: Training Loss', fontsize=18)
plt.show()

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
report = classification_report(y_test, y_pred)
for cls, metrics in report.items():
    print(f'  {cls:15s}: {metrics}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel('Predicted', fontsize=13)
plt.ylabel('True', fontsize=13)
plt.title('Logistic Regression: Confusion Matrix', fontsize=16)
plt.show()

In [ ]:
# Predicted probability distribution
plt.figure(figsize=(10, 6))
plt.hist(y_proba[y_test == 0], bins=20, alpha=0.6, color='steelblue', label='True: No')
plt.hist(y_proba[y_test == 1], bins=20, alpha=0.6, color='salmon', label='True: Yes')
plt.axvline(0.5, color='black', linestyle='--', label='Decision boundary')
plt.xlabel('Predicted Probability', fontsize=15)
plt.ylabel('Count', fontsize=15)
plt.title('Predicted Probability Distribution', fontsize=18)
plt.legend(fontsize=13)
plt.show()

## Interpretation

- The sigmoid function maps the linear output to a probability. points above 0.5 are classified as positive.
- The cross-entropy loss penalizes **confident wrong predictions** more heavily than uncertain ones.
- The probability histogram shows separation: blue (true negatives) should cluster near 0, red (true positives) near 1.